# CCHP Kaggle 入口 `main.ipynb`

用于在 Kaggle 直接运行 `cchp_physical_env` 的训练/评估流程，并支持导入上次导出的 checkpoint 续训。

## Add data 约定
- 代码数据集：包含 `pyproject.toml` + `src/cchp_physical_env/`
- 数据数据集：包含 `data/processed/cchp_main_15min_2024.csv` 与 `data/processed/cchp_main_15min_2025.csv`

按 cell 顺序执行即可。

说明：
- Notebook 会生成一个 runtime config（Notebook overrides 优先级最高），并统一用 `--env-config <runtime.yaml>` 调用 CLI。
- 常规 baseline / sequence / SB3 仍走统一 `train` / `eval`。
- `DPAR` 路径走显式 `hybrid-train` / `hybrid-eval`，需要在参数区提供 `dpar_overrides.dqn_checkpoint` 与 `dpar_overrides.pafc_checkpoint`。
- 若 `training.sb3_enabled=true`，则 `train` 会优先走 SB3（PPO/SAC/TD3/DDPG/DQN），`eval --checkpoint <baseline_policy.json>` 会自动识别 SB3 checkpoint 并走 SB3 评估。
- SB3 路径下观测为 windowed 的 `Box(K,D)`，并通过自定义 `features_extractor` 把 `(K,D)` 编码成特征向量；因此可用 `training.sb3_backbone`/`training.sb3_history_steps` 组合实现 “SAC+Transformer / TD3+Mamba” 等对比。


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Has /kaggle/input:', Path('/kaggle/input').exists())
print('Has /kaggle/working:', Path('/kaggle/working').exists())
print('CPU count:', os.cpu_count())

for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    if key in os.environ:
        print(f'{key}:', os.environ.get(key))


## 0) Notebook 参数区（只改这一格）

Notebook 参数优先级最高：后续会生成 runtime config，CLI 全部使用 runtime config（Notebook > base config）。

In [ ]:
from dataclasses import dataclass, field


@dataclass
class RunCfg:
    # Kaggle 数据集目录（你上传的数据集路径）
    kaggle_codeset_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-code'
    kaggle_dataset_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-data'

    # 上一次导出的输出数据集（用于断点续训）
    # 为空则自动扫描 /kaggle/input
    kaggle_prev_output_dir: str = '/kaggle/input/datasets/tttwwwjjj/cchp-continue'

    # 同步开关
    sync_code_to_cwd: bool = True
    sync_data_to_repo: bool = True

    # 运行配置
    env_config_relpath: str = 'src/cchp_physical_env/config/config.yaml'
    run_root: str = 'runs'
    run_prefix: str = 'kaggle_cchp'

    # DPAR / hybrid 模式
    # - False：按原 notebook 走 baseline / sequence / SB3
    # - True：显式走 `hybrid-train` / `hybrid-eval`
    dpar_enabled: bool = False

    # 断点续训
    resume_enabled: bool = True
    # 注意：若你要跑新的 SB3 或 DPAR 组装对比，建议把 skip_train_when_resuming 设为 False
    skip_train_when_resuming: bool = True
    resume_sequence_via_api: bool = True

    # 是否把上次 runs 目录拷贝到当前 runs（通常不需要）
    resume_copy_runs: bool = False

    # 安装最小兜底依赖（避免 Kaggle 环境缺包导致 import 失败）
    # - SB3 依赖（stable-baselines3/gymnasium）在 sb3_enabled=true 或 dpar_enabled=true 时需要
    # - 若使用 mamba（含 sb3_backbone=mamba / sequence_adapter=mamba），实现来自 transformers，无需额外安装 mamba-ssm
    install_extra_deps: tuple[str, ...] = (
        'numpy',
        'pandas',
        'pyyaml',
        'pyomo',
        'tespy',
        'CoolProp',
        'transformers',
        'torch',
        # SB3 可选依赖（需要时再打开）：
        # 'stable-baselines3',
        # 'gymnasium',
    )

    # ------------------------------
    # Notebook 只改这一格：覆盖参数
    # ------------------------------

    env_overrides: dict[str, object] = field(default_factory=lambda: {
        # 'constraint_mode': 'physics_in_loop',
        # 'physics_backend': 'tespy',
    })

    # DPAR / hybrid 参数
    # - `dqn_checkpoint`：高层 DQN 的 `sb3_policy` json
    # - `pafc_checkpoint`：低层 PAFC checkpoint（`.pt` 或 `pafc_td3_actor.json`）
    dpar_overrides: dict[str, object] = field(default_factory=lambda: {
        'dqn_checkpoint': '',
        'pafc_checkpoint': '',
        'discrete_hold_steps': 1,
        'residual_scale': 1.0,
        'refine_action_keys': 'u_boiler',
        'refine_action_scales': '',
        'model_source': 'best',
        'stochastic_dqn': False,
        'use_safe_residual': False,
    })

    # PPO warm-start（easy_rule 行为克隆预热：只预热 u_boiler/u_ech）
    sb3_ppo_warm_start_enabled: bool = True
    sb3_ppo_warm_start_samples: int = 16_384
    sb3_ppo_warm_start_epochs: int = 4
    sb3_ppo_warm_start_batch_size: int = 256
    sb3_ppo_warm_start_lr: float = 1e-4

    # 连续动作 hybrid residual（PPO/SAC/TD3/DDPG 共用）
    sb3_residual_enabled: bool = True
    sb3_residual_policy: str = 'rule'
    sb3_residual_scale: float = 0.25

    # Off-policy prefill（SAC/TD3/DDPG 用规则策略预填充 replay buffer）
    sb3_offpolicy_prefill_enabled: bool = True
    sb3_offpolicy_prefill_policy: str = 'rule'
    sb3_offpolicy_prefill_steps: int = 20000

    # best checkpoint 选择：先过可靠性门槛，再按 reward 选最优
    sb3_best_gate_enabled: bool = True
    sb3_best_gate_electric_min: float = 1.0
    sb3_best_gate_heat_min: float = 0.99
    sb3_best_gate_cool_min: float = 0.99

    # plateau 控制：先降 LR 做 fine-tune，若仍停滞则提前停止
    sb3_plateau_control_enabled: bool = True
    sb3_plateau_patience_evals: int = 10
    sb3_plateau_lr_decay_factor: float = 0.5
    sb3_plateau_min_lr: float = 5e-5
    sb3_plateau_early_stop_patience_evals: int = 999

    # shaping：ABS deadzone / GT / backup idle
    abs_deadzone_gate_th: float = 0.15
    abs_deadzone_u_th: float = 0.10
    penalty_gt_toggle: float = 100.0
    penalty_gt_delta_mw: float = 5.0
    penalty_idle_heat_backup: float = 50.0
    penalty_idle_cool_backup: float = 50.0

    # training 覆盖参数（Notebook 优先级最高，会覆盖 config.yaml 对应项）
    #
    # 路由：
    # 1) dpar_enabled=true -> notebook 显式调用 hybrid-train / hybrid-eval
    # 2) sb3_enabled=true  -> `train` 优先进入 SB3 路由（PPO/SAC/TD3/DDPG/DQN）
    # 3) sb3_enabled=false 且 policy=sequence_rule 且 sequence_adapter in {mlp, transformer, mamba}
    #    -> 进入内置 PPO-style sequence trainer
    # 4) 其他情况 -> baseline 路由（rule / easy_rule / random / sequence_rule+rule）
    training_overrides: dict[str, object] = field(default_factory=lambda: {
        'seed': 42,
        'policy': 'sequence_rule',
        'sequence_adapter': 'transformer',
        'history_steps': 32,
        'episode_days': 14,
        'episodes': 800,
        'train_steps': 409600,
        'batch_size': 256,
        'update_epochs': 8,
        'lr': 0.0001,
        'device': 'auto',

        'sb3_enabled': True,
        'sb3_algo': 'ddpg',
        'sb3_backbone': 'mlp',
        'sb3_history_steps': 32,
        'sb3_total_timesteps': 2000000,
        'sb3_n_envs': 4,
        'sb3_learning_rate': 0.0001,
        'sb3_batch_size': 256,
        'sb3_gamma': 0.99,

        'sb3_vec_norm_obs': True,
        'sb3_vec_norm_reward': True,

        'sb3_eval_freq': 50000,
        'sb3_eval_episode_days': 14,

        'sb3_ppo_n_steps': 2048,
        'sb3_ppo_gae_lambda': 0.95,
        'sb3_ppo_ent_coef': 0.0,
        'sb3_ppo_clip_range': 0.2,

        'sb3_learning_starts': 20000,
        'sb3_train_freq': 4,
        'sb3_gradient_steps': 4,
        'sb3_tau': 0.005,
        'sb3_action_noise_std': 0.05,
        'sb3_buffer_size': 100000,
        'sb3_optimize_memory_usage': True,

        'sb3_eval_window_pool_size': 12,
        'sb3_eval_window_count': 4,
        'sb3_residual_enabled': True,
        'sb3_residual_policy': 'rule',
        'sb3_residual_scale': 0.25,
        'sb3_best_gate_enabled': True,
        'sb3_best_gate_electric_min': 1.0,
        'sb3_best_gate_heat_min': 0.99,
        'sb3_best_gate_cool_min': 0.99,
        'sb3_plateau_control_enabled': True,
        'sb3_plateau_patience_evals': 10,
        'sb3_plateau_lr_decay_factor': 0.5,
        'sb3_plateau_min_lr': 5e-5,
        'sb3_plateau_early_stop_patience_evals': 999,
    })

    # 评估阶段 CLI 覆盖（只放非 seed 类参数；seed 始终复用 training_overrides['seed']）
    eval_cli_overrides: dict[str, object] = field(default_factory=lambda: {
        # 'model_source': 'best',
    })

    def __post_init__(self) -> None:
        self.env_overrides.setdefault('abs_deadzone_gate_th', float(self.abs_deadzone_gate_th))
        self.env_overrides.setdefault('abs_deadzone_u_th', float(self.abs_deadzone_u_th))
        self.env_overrides.setdefault('penalty_gt_toggle', float(self.penalty_gt_toggle))
        self.env_overrides.setdefault('penalty_gt_delta_mw', float(self.penalty_gt_delta_mw))
        self.env_overrides.setdefault('penalty_idle_heat_backup', float(self.penalty_idle_heat_backup))
        self.env_overrides.setdefault('penalty_idle_cool_backup', float(self.penalty_idle_cool_backup))

        self.training_overrides.setdefault('sb3_ppo_warm_start_enabled', bool(self.sb3_ppo_warm_start_enabled))
        self.training_overrides.setdefault('sb3_ppo_warm_start_samples', int(self.sb3_ppo_warm_start_samples))
        self.training_overrides.setdefault('sb3_ppo_warm_start_epochs', int(self.sb3_ppo_warm_start_epochs))
        self.training_overrides.setdefault('sb3_ppo_warm_start_batch_size', int(self.sb3_ppo_warm_start_batch_size))
        self.training_overrides.setdefault('sb3_ppo_warm_start_lr', float(self.sb3_ppo_warm_start_lr))
        self.training_overrides.setdefault('sb3_residual_enabled', bool(self.sb3_residual_enabled))
        self.training_overrides.setdefault('sb3_residual_policy', str(self.sb3_residual_policy).strip())
        self.training_overrides.setdefault('sb3_residual_scale', float(self.sb3_residual_scale))
        self.training_overrides.setdefault('sb3_offpolicy_prefill_enabled', bool(self.sb3_offpolicy_prefill_enabled))
        self.training_overrides.setdefault('sb3_offpolicy_prefill_steps', int(self.sb3_offpolicy_prefill_steps))
        self.training_overrides.setdefault('sb3_offpolicy_prefill_policy', str(self.sb3_offpolicy_prefill_policy).strip())
        self.training_overrides.setdefault('sb3_best_gate_enabled', bool(self.sb3_best_gate_enabled))
        self.training_overrides.setdefault('sb3_best_gate_electric_min', float(self.sb3_best_gate_electric_min))
        self.training_overrides.setdefault('sb3_best_gate_heat_min', float(self.sb3_best_gate_heat_min))
        self.training_overrides.setdefault('sb3_best_gate_cool_min', float(self.sb3_best_gate_cool_min))
        self.training_overrides.setdefault('sb3_plateau_control_enabled', bool(self.sb3_plateau_control_enabled))
        self.training_overrides.setdefault('sb3_plateau_patience_evals', int(self.sb3_plateau_patience_evals))
        self.training_overrides.setdefault('sb3_plateau_lr_decay_factor', float(self.sb3_plateau_lr_decay_factor))
        self.training_overrides.setdefault('sb3_plateau_min_lr', float(self.sb3_plateau_min_lr))
        self.training_overrides.setdefault('sb3_plateau_early_stop_patience_evals', int(self.sb3_plateau_early_stop_patience_evals))

        self.dpar_overrides.setdefault('dqn_checkpoint', '')
        self.dpar_overrides.setdefault('pafc_checkpoint', '')
        self.dpar_overrides.setdefault('discrete_hold_steps', 1)
        self.dpar_overrides.setdefault('residual_scale', 1.0)
        self.dpar_overrides.setdefault('refine_action_keys', 'u_boiler')
        self.dpar_overrides.setdefault('refine_action_scales', '')
        self.dpar_overrides.setdefault('model_source', 'best')
        self.dpar_overrides.setdefault('stochastic_dqn', False)
        self.dpar_overrides.setdefault('use_safe_residual', False)

        self.dpar_overrides['dqn_checkpoint'] = str(self.dpar_overrides.get('dqn_checkpoint', '')).strip()
        self.dpar_overrides['pafc_checkpoint'] = str(self.dpar_overrides.get('pafc_checkpoint', '')).strip()
        self.dpar_overrides['discrete_hold_steps'] = int(self.dpar_overrides.get('discrete_hold_steps', 1))
        self.dpar_overrides['residual_scale'] = float(self.dpar_overrides.get('residual_scale', 1.0))
        self.dpar_overrides['refine_action_keys'] = str(self.dpar_overrides.get('refine_action_keys', 'u_boiler')).strip()
        self.dpar_overrides['refine_action_scales'] = str(self.dpar_overrides.get('refine_action_scales', '')).strip()
        self.dpar_overrides['model_source'] = str(self.dpar_overrides.get('model_source', 'best')).strip().lower()
        self.dpar_overrides['stochastic_dqn'] = bool(self.dpar_overrides.get('stochastic_dqn', False))
        self.dpar_overrides['use_safe_residual'] = bool(self.dpar_overrides.get('use_safe_residual', False))

        if self.dpar_enabled:
            self.training_overrides['policy'] = 'hybrid_pafc'
            self.training_overrides['sb3_enabled'] = False
            self.training_overrides.setdefault('sequence_adapter', 'rule')


rc = RunCfg()
print(rc)


## 1) 同步代码 + 安装

- 扫描 `/kaggle/input` 中包含 `pyproject.toml` + `src/cchp_physical_env` 的目录
- 同步到当前可写目录
- `%pip install -e . --no-deps`
- 注入 `./src` 到 `sys.path`，保证 `import cchp_physical_env` 稳定

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def _run_cmd(cmd: list[str]) -> None:
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def _rsync_or_copy(src: Path, dst: Path) -> None:
    src = src.resolve()
    dst = dst.resolve()
    if shutil.which('rsync'):
        _run_cmd(['rsync', '-a', '--info=progress2', f'{src.as_posix()}/', f'{dst.as_posix()}/'])
        return

    print('rsync not found, fallback to shutil.copytree')
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(item, target)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)


def _find_codeset_dir(preferred: str) -> Path:
    p = Path(preferred)
    if (p / 'pyproject.toml').exists() and (p / 'src' / 'cchp_physical_env').exists():
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for pyproj in root.rglob('pyproject.toml'):
        cand = pyproj.parent
        if (cand / 'src' / 'cchp_physical_env').exists():
            hits.append(cand)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        chosen = sorted(hits, key=lambda x: x.as_posix())[0]
        print('Multiple code datasets detected. Using:', chosen)
        return chosen

    return p


def _try_install_conda_forge(packages: list[str]) -> bool:
    installer = None
    if shutil.which('mamba'):
        installer = 'mamba'
    elif shutil.which('conda'):
        installer = 'conda'

    if installer is None:
        print('conda/mamba not found, skip conda-forge fallback')
        return False

    cmd = [installer, 'install', '-y', '-c', 'conda-forge', *packages]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    return True


code_src = _find_codeset_dir(rc.kaggle_codeset_dir)
assert code_src.exists(), f'missing code dataset: {code_src}'
print('code_src:', code_src)

if rc.sync_code_to_cwd:
    _rsync_or_copy(code_src, Path.cwd())

assert Path('pyproject.toml').exists(), 'missing pyproject.toml after sync'
assert Path('src/cchp_physical_env').exists(), 'missing src/cchp_physical_env after sync'

src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print('sys.executable:', sys.executable)
print('sys.path[0]:', sys.path[0])

ip = get_ipython()
assert ip is not None

ip.run_line_magic('pip', 'install -q -e . --no-deps')

if rc.install_extra_deps:
    ip.run_line_magic('pip', f"install -q {' '.join(rc.install_extra_deps)}")

try:
    import tespy  # noqa: F401
    import CoolProp  # noqa: F401
    print('tespy/CoolProp import: OK')
except Exception as error:
    print('tespy/CoolProp import failed after pip install:', repr(error))
    ok = _try_install_conda_forge(['tespy', 'coolprop'])
    if ok:
        import tespy  # noqa: F401
        import CoolProp  # noqa: F401
        print('tespy/CoolProp import: OK (conda-forge)')
    else:
        raise RuntimeError(
            'tespy 安装失败。若当前 kernel 为 Python 3.12，建议切换到 Python 3.11/3.10 再重试；'
            '或手动安装可用的 CoolProp/tespy 二进制包。'
        )

sb3_enabled = bool(rc.training_overrides.get('sb3_enabled', False))
dpar_enabled = bool(getattr(rc, 'dpar_enabled', False))
sb3_backbone = str(rc.training_overrides.get('sb3_backbone', 'mlp')).strip().lower()
need_sb3_stack = sb3_enabled or dpar_enabled
if need_sb3_stack:
    ip.run_line_magic('pip', 'install -q stable-baselines3 gymnasium')
    if sb3_enabled and sb3_backbone == 'mamba':
        print('mamba backbone is provided by transformers; no extra mamba-ssm install is needed')

import cchp_physical_env
print('cchp_physical_env imported from:', cchp_physical_env.__file__)
print('sb3_enabled:', sb3_enabled)
print('dpar_enabled:', dpar_enabled)
print('sb3_backbone:', sb3_backbone)


## 2) 同步数据到 `./data/`

- 扫描 `/kaggle/input` 中包含 `data/processed/*.csv` 的目录
- 同步到 `./data/`
- 强校验 `cchp_main_15min_2024.csv` 与 `cchp_main_15min_2025.csv`

In [ ]:
from pathlib import Path


def _find_data_root(preferred: str) -> Path:
    p = Path(preferred)
    if (p / 'data' / 'processed').exists() and list((p / 'data' / 'processed').glob('*.csv')):
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for processed in root.rglob('data/processed'):
        if processed.is_dir() and list(processed.glob('*.csv')):
            hits.append(processed.parent.parent)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        chosen = sorted(hits, key=lambda x: x.as_posix())[0]
        print('Multiple data datasets detected. Using:', chosen)
        return chosen

    return p


data_root = _find_data_root(rc.kaggle_dataset_dir)
assert data_root.exists(), f'missing data dataset: {data_root}'
print('data_root:', data_root)

if rc.sync_data_to_repo:
    Path('data').mkdir(parents=True, exist_ok=True)
    _rsync_or_copy(data_root / 'data', Path('data'))

train_csv = Path('data/processed/cchp_main_15min_2024.csv')
eval_csv = Path('data/processed/cchp_main_15min_2025.csv')
assert train_csv.exists(), train_csv
assert eval_csv.exists(), eval_csv
print('train_csv:', train_csv)
print('eval_csv:', eval_csv)


## 3) 生成 runtime config（Notebook 优先级最高）

- 读取 base `config.yaml`
- 仅用 `env_overrides` / `training_overrides` 覆盖你要改的键
- 生成 runtime yaml，后续 CLI 全部使用 runtime yaml

In [ ]:
from pathlib import Path
import copy
import json
import yaml

base_cfg_path = Path(rc.env_config_relpath)
assert base_cfg_path.exists(), base_cfg_path

with base_cfg_path.open('r', encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)

assert isinstance(base_cfg, dict), 'config.yaml must be a mapping'
assert isinstance(base_cfg.get('env'), dict), 'config.yaml missing env block'
assert isinstance(base_cfg.get('training'), dict), 'config.yaml missing training block'

runtime_cfg = copy.deepcopy(base_cfg)


def _apply_overrides(block_name: str, block: dict, overrides: dict[str, object]) -> None:
    for key, value in overrides.items():
        if key not in block:
            raise KeyError(f'unknown key in notebook overrides: {block_name}.{key}')
        block[key] = value


_apply_overrides('env', runtime_cfg['env'], rc.env_overrides)
_apply_overrides('training', runtime_cfg['training'], rc.training_overrides)

runtime_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
runtime_cfg_path = runtime_dir / f'{rc.run_prefix}_runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

print('runtime config:', runtime_cfg_path)
print('env_overrides:', json.dumps(rc.env_overrides, ensure_ascii=False, indent=2))
print('training_overrides:', json.dumps(rc.training_overrides, ensure_ascii=False, indent=2))


In [ ]:
from __future__ import annotations

from pathlib import Path


def _format_cli(parts: list[str]) -> str:
    return ' \
  '.join(parts)


def _display_value(value: object, placeholder: str = '<required>') -> str:
    text = str(value).strip()
    return text if text else placeholder


runtime_cfg_cli = Path(runtime_cfg_path).as_posix() if 'runtime_cfg_path' in globals() else rc.env_config_relpath
run_root_cli = str(Path(rc.run_root).as_posix())
dpar_enabled = bool(getattr(rc, 'dpar_enabled', False))
dpar_cfg = dict(getattr(rc, 'dpar_overrides', {}))

cmd_summary = _format_cli([
    'python -m cchp_physical_env summary',
    f'--env-config {runtime_cfg_cli}',
])

cmd_train_unified = _format_cli([
    'python -m cchp_physical_env train',
    f'--env-config {runtime_cfg_cli}',
    f'--run-root {run_root_cli}',
    f"--policy {rc.training_overrides['policy']}",
    f"--sequence-adapter {rc.training_overrides['sequence_adapter']}",
    f"--history-steps {rc.training_overrides['history_steps']}",
    f"--episode-days {rc.training_overrides['episode_days']}",
    f"--episodes {rc.training_overrides['episodes']}",
    f"--train-steps {rc.training_overrides['train_steps']}",
    f"--batch-size {rc.training_overrides['batch_size']}",
    f"--update-epochs {rc.training_overrides['update_epochs']}",
    f"--lr {rc.training_overrides['lr']}",
    f"--device {rc.training_overrides['device']}",
    f"--seed {rc.training_overrides['seed']}",
])

cmd_train_unified_sb3 = _format_cli([
    'python -m cchp_physical_env train',
    f'--env-config {runtime_cfg_cli}',
    f'--run-root {run_root_cli}',
    '--sb3-enabled',
    f"--sb3-algo {rc.training_overrides['sb3_algo']}",
    f"--sb3-backbone {rc.training_overrides['sb3_backbone']}",
    f"--sb3-history-steps {rc.training_overrides['sb3_history_steps']}",
    f"--sb3-total-timesteps {rc.training_overrides['sb3_total_timesteps']}",
    f"--episode-days {rc.training_overrides['episode_days']}",
    f"--sb3-n-envs {rc.training_overrides['sb3_n_envs']}",
    f"--sb3-learning-rate {rc.training_overrides['sb3_learning_rate']}",
    f"--sb3-batch-size {rc.training_overrides['sb3_batch_size']}",
    f"--sb3-gamma {rc.training_overrides['sb3_gamma']}",
    '--sb3-vec-norm-obs',
    '--sb3-vec-norm-reward',
    f"--sb3-eval-freq {rc.training_overrides['sb3_eval_freq']}",
    f"--sb3-eval-episode-days {rc.training_overrides['sb3_eval_episode_days']}",
    f"--sb3-eval-window-pool-size {rc.training_overrides['sb3_eval_window_pool_size']}",
    f"--sb3-eval-window-count {rc.training_overrides['sb3_eval_window_count']}",
    '--sb3-residual-enabled',
    f"--sb3-residual-policy {rc.training_overrides['sb3_residual_policy']}",
    f"--sb3-residual-scale {rc.training_overrides['sb3_residual_scale']}",
    f"--sb3-ppo-n-steps {rc.training_overrides['sb3_ppo_n_steps']}",
    f"--sb3-ppo-gae-lambda {rc.training_overrides['sb3_ppo_gae_lambda']}",
    f"--sb3-ppo-ent-coef {rc.training_overrides['sb3_ppo_ent_coef']}",
    f"--sb3-ppo-clip-range {rc.training_overrides['sb3_ppo_clip_range']}",
    f"--sb3-learning-starts {rc.training_overrides['sb3_learning_starts']}",
    f"--sb3-train-freq {rc.training_overrides['sb3_train_freq']}",
    f"--sb3-gradient-steps {rc.training_overrides['sb3_gradient_steps']}",
    f"--sb3-tau {rc.training_overrides['sb3_tau']}",
    f"--sb3-action-noise-std {rc.training_overrides['sb3_action_noise_std']}",
    f"--sb3-buffer-size {rc.training_overrides['sb3_buffer_size']}",
    '--sb3-optimize-memory-usage',
    '--sb3-best-gate-enabled',
    f"--sb3-best-gate-electric-min {rc.training_overrides['sb3_best_gate_electric_min']}",
    f"--sb3-best-gate-heat-min {rc.training_overrides['sb3_best_gate_heat_min']}",
    f"--sb3-best-gate-cool-min {rc.training_overrides['sb3_best_gate_cool_min']}",
    '--sb3-plateau-control-enabled',
    f"--sb3-plateau-patience-evals {rc.training_overrides['sb3_plateau_patience_evals']}",
    f"--sb3-plateau-lr-decay-factor {rc.training_overrides['sb3_plateau_lr_decay_factor']}",
    f"--sb3-plateau-min-lr {rc.training_overrides['sb3_plateau_min_lr']}",
    f"--sb3-plateau-early-stop-patience-evals {rc.training_overrides['sb3_plateau_early_stop_patience_evals']}",
    f"--device {rc.training_overrides['device']}",
    f"--seed {rc.training_overrides['seed']}",
])

cmd_sb3_train_explicit = _format_cli([
    'python -m cchp_physical_env sb3-train',
    f'--env-config {runtime_cfg_cli}',
    f'--run-root {run_root_cli}',
    f"--algo {rc.training_overrides['sb3_algo']}",
    f"--backbone {rc.training_overrides['sb3_backbone']}",
    f"--history-steps {rc.training_overrides['sb3_history_steps']}",
    f"--total-timesteps {rc.training_overrides['sb3_total_timesteps']}",
    f"--episode-days {rc.training_overrides['episode_days']}",
    f"--n-envs {rc.training_overrides['sb3_n_envs']}",
    f"--learning-rate {rc.training_overrides['sb3_learning_rate']}",
    f"--batch-size {rc.training_overrides['sb3_batch_size']}",
    f"--gamma {rc.training_overrides['sb3_gamma']}",
    '--vec-norm-obs',
    '--vec-norm-reward',
    f"--eval-freq {rc.training_overrides['sb3_eval_freq']}",
    f"--eval-episode-days {rc.training_overrides['sb3_eval_episode_days']}",
    f"--eval-window-pool-size {rc.training_overrides['sb3_eval_window_pool_size']}",
    f"--eval-window-count {rc.training_overrides['sb3_eval_window_count']}",
    '--residual-enabled',
    f"--residual-policy {rc.training_overrides['sb3_residual_policy']}",
    f"--residual-scale {rc.training_overrides['sb3_residual_scale']}",
    f"--ppo-n-steps {rc.training_overrides['sb3_ppo_n_steps']}",
    f"--ppo-gae-lambda {rc.training_overrides['sb3_ppo_gae_lambda']}",
    f"--ppo-ent-coef {rc.training_overrides['sb3_ppo_ent_coef']}",
    f"--ppo-clip-range {rc.training_overrides['sb3_ppo_clip_range']}",
    f"--learning-starts {rc.training_overrides['sb3_learning_starts']}",
    f"--train-freq {rc.training_overrides['sb3_train_freq']}",
    f"--gradient-steps {rc.training_overrides['sb3_gradient_steps']}",
    f"--tau {rc.training_overrides['sb3_tau']}",
    f"--action-noise-std {rc.training_overrides['sb3_action_noise_std']}",
    f"--buffer-size {rc.training_overrides['sb3_buffer_size']}",
    '--optimize-memory-usage',
    '--best-gate-enabled',
    f"--best-gate-electric-min {rc.training_overrides['sb3_best_gate_electric_min']}",
    f"--best-gate-heat-min {rc.training_overrides['sb3_best_gate_heat_min']}",
    f"--best-gate-cool-min {rc.training_overrides['sb3_best_gate_cool_min']}",
    '--plateau-control-enabled',
    f"--plateau-patience-evals {rc.training_overrides['sb3_plateau_patience_evals']}",
    f"--plateau-lr-decay-factor {rc.training_overrides['sb3_plateau_lr_decay_factor']}",
    f"--plateau-min-lr {rc.training_overrides['sb3_plateau_min_lr']}",
    f"--plateau-early-stop-patience-evals {rc.training_overrides['sb3_plateau_early_stop_patience_evals']}",
    f"--device {rc.training_overrides['device']}",
    f"--seed {rc.training_overrides['seed']}",
])

cmd_eval_unified = _format_cli([
    'python -m cchp_physical_env eval',
    f'--env-config {runtime_cfg_cli}',
    '--checkpoint <path/to/baseline_policy.json>',
    f"--seed {rc.training_overrides['seed']}",
    f"--device {rc.training_overrides['device']}",
    '--model-source best',
])

cmd_sb3_eval_explicit = _format_cli([
    'python -m cchp_physical_env sb3-eval',
    '--run-dir <path/to/run_dir>',
    '--checkpoint <path/to/run_dir/checkpoints/baseline_policy.json>',
    f"--seed {rc.training_overrides['seed']}",
    f"--device {rc.training_overrides['device']}",
    '--model-source best',
])

hybrid_train_parts = [
    'python -m cchp_physical_env hybrid-train',
    f'--env-config {runtime_cfg_cli}',
    f'--run-root {run_root_cli}',
    f"--dqn-checkpoint {_display_value(dpar_cfg.get('dqn_checkpoint', ''))}",
    f"--pafc-checkpoint {_display_value(dpar_cfg.get('pafc_checkpoint', ''))}",
    f"--discrete-hold-steps {int(dpar_cfg.get('discrete_hold_steps', 1))}",
    f"--residual-scale {float(dpar_cfg.get('residual_scale', 1.0))}",
    f"--refine-action-keys {_display_value(dpar_cfg.get('refine_action_keys', 'u_boiler'), 'u_boiler')}",
    f"--model-source {_display_value(dpar_cfg.get('model_source', 'best'), 'best')}",
    f"--device {rc.training_overrides['device']}",
    f"--seed {rc.training_overrides['seed']}",
]
if str(dpar_cfg.get('refine_action_scales', '')).strip():
    hybrid_train_parts.append(f"--refine-action-scales {str(dpar_cfg.get('refine_action_scales', '')).strip()}")
if bool(dpar_cfg.get('stochastic_dqn', False)):
    hybrid_train_parts.append('--stochastic-dqn')
if bool(dpar_cfg.get('use_safe_residual', False)):
    hybrid_train_parts.append('--use-safe-residual')
cmd_hybrid_train = _format_cli(hybrid_train_parts)

cmd_hybrid_eval = _format_cli([
    'python -m cchp_physical_env hybrid-eval',
    '--run-dir <path/to/run_dir>',
    '--checkpoint <path/to/run_dir/checkpoints/hybrid_pafc_policy.json>',
    f"--env-config {runtime_cfg_cli}",
    f"--device {rc.training_overrides['device']}",
    f"--seed {rc.training_overrides['seed']}",
])

print('\n===== CLI Commands (Copy & Run) =====\n')
print('[summary]')
print(cmd_summary)
print('\n[train - unified baseline/sequence]')
print(cmd_train_unified)
print('\n[train - unified SB3 route]')
print(cmd_train_unified_sb3)
print('\n[sb3-train - explicit]')
print(cmd_sb3_train_explicit)
print('\n[eval - unified]')
print(cmd_eval_unified)
print('\n[sb3-eval - explicit]')
print(cmd_sb3_eval_explicit)
print('\n[hybrid-train - DPAR / explicit]')
print(cmd_hybrid_train)
print('\n[hybrid-eval - DPAR / explicit]')
print(cmd_hybrid_eval)
print('\nActive notebook mode:', 'DPAR' if dpar_enabled else 'standard')


## 4) 断点续训：自动定位上次 checkpoint

- 扫描 `/kaggle/input`（或你指定的 `kaggle_prev_output_dir`）
- 常规模式定位最新的 `baseline_policy.json`
- `DPAR` 模式定位最新的 `hybrid_pafc_policy.json`

说明：

- baseline（rule/random）：可直接复用 checkpoint，跳过最小 train
- sequence_rule（transformer/mamba）：可选尝试加载上次模型权重 warm-start；失败自动回退 CLI train
- SB3（ppo/sac/td3/ddpg/dqn）：若本次 `training.sb3_enabled=true`，默认会跑一次新的 SB3 train（避免被旧 checkpoint 覆盖实验口径）
- DPAR（hybrid_pafc）：若本次 `dpar_enabled=true`，默认复用已有 `hybrid_pafc_policy.json`；若关闭 `skip_train_when_resuming`，则重新执行一次 hybrid 组装

最终统一保存一个“当前用于评估的 checkpoint 路径”，后续 cell 会按模式选择 `eval` 或 `hybrid-eval`。


In [ ]:
from pathlib import Path


def _detect_prev_output_dir(preferred: str) -> Path | None:
    if preferred:
        p = Path(preferred)
        if p.exists():
            return p

    continue_dir = Path('/kaggle/input/datasets/tttwwwjjj/cchp-continue')
    if continue_dir.exists():
        return continue_dir

    root = Path('/kaggle/input')
    if not root.exists():
        return None

    candidates: list[Path] = []
    for d in root.rglob('*'):
        if not d.is_dir():
            continue
        if (d / 'kaggle_exports').exists() or (d / 'runs').exists() or (d / 'checkpoints').exists():
            candidates.append(d)

    if not candidates:
        return None

    return sorted(candidates, key=lambda x: x.as_posix())[0]


def _collect_checkpoint_jsons(search_root: Path, checkpoint_names: tuple[str, ...]) -> list[Path]:
    if not search_root.exists():
        return []
    hits: list[Path] = []
    for checkpoint_name in checkpoint_names:
        hits.extend([p for p in search_root.rglob(checkpoint_name) if p.is_file()])
    deduped = sorted(set(hits), key=lambda p: p.stat().st_mtime, reverse=True)
    return deduped


resume_checkpoint_json: Path | None = None
prev_output_dir: Path | None = None
checkpoint_names = ('hybrid_pafc_policy.json',) if bool(getattr(rc, 'dpar_enabled', False)) else ('baseline_policy.json',)

if rc.resume_enabled:
    prev_output_dir = _detect_prev_output_dir(rc.kaggle_prev_output_dir)
    print('prev_output_dir:', prev_output_dir)
    print('checkpoint_names:', checkpoint_names)
    if prev_output_dir is not None:
        ckpt_hits = _collect_checkpoint_jsons(prev_output_dir, checkpoint_names)
        if ckpt_hits:
            resume_checkpoint_json = ckpt_hits[0]
            print('resume checkpoint found:', resume_checkpoint_json)

            if rc.resume_copy_runs:
                run_root = Path(rc.run_root)
                run_root.mkdir(parents=True, exist_ok=True)
                src_runs = prev_output_dir / 'runs'
                if src_runs.exists() and src_runs.is_dir():
                    _rsync_or_copy(src_runs, run_root)
        else:
            print(f'No checkpoint json found under previous output dir for: {checkpoint_names}')


## 5) CLI：`summary` / `train` / `eval`

执行顺序：
- `summary`
- 最小 `train`（或 resume 跳过/续训 / hybrid 组装）
- `eval`

标准 SB3 + 序列骨干（支持 “SAC+Transformer”）：
- 方式 A（推荐，走统一 `train` 路由）：在 Notebook 参数区把
  - `training_overrides.sb3_enabled=True`
  - `training_overrides.sb3_algo='sac'`
  - `training_overrides.sb3_backbone='transformer'`
  - `training_overrides.sb3_history_steps=16`
  然后直接运行本节 cell。
- 方式 B（显式命令，通常用于本地调试）：
  - `python -m cchp_physical_env sb3-train --algo sac --backbone transformer --history-steps 16 ...`

DPAR：
- 在参数区设置 `dpar_enabled=True`
- 填入 `dpar_overrides.dqn_checkpoint` 与 `dpar_overrides.pafc_checkpoint`
- 然后直接运行本节 cell，notebook 会自动走 `hybrid-train` / `hybrid-eval`

注意：
- SB3 训练会使用 windowed observation 的 `Box(K,D)`，并通过自定义 `features_extractor` 将 `(K,D)` 编码为特征向量。
- 若启用 SB3，默认不会复用旧的 `baseline_policy.json` 直接跳过训练（避免实验口径被旧 checkpoint 污染）。
- 若启用 DPAR，默认复用旧的 `hybrid_pafc_policy.json` 跳过重新组装；若要重新组装，请把 `skip_train_when_resuming=False`。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys


def _run_cli(args: list[str]) -> None:
    env = os.environ.copy()
    py_src = str((Path.cwd() / 'src').resolve())
    env['PYTHONPATH'] = py_src + (os.pathsep + env['PYTHONPATH'] if 'PYTHONPATH' in env else '')
    cmd = [sys.executable, '-m', 'cchp_physical_env', *args]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True, env=env)


def _latest_checkpoint_from_run_root(run_root: Path, checkpoint_name: str = 'baseline_policy.json') -> Path:
    if not run_root.exists():
        raise FileNotFoundError(f'run_root not found: {run_root}')
    run_dirs = [p for p in run_root.iterdir() if p.is_dir()]
    if not run_dirs:
        raise FileNotFoundError(f'no run dirs under: {run_root}')
    latest = sorted(run_dirs, key=lambda p: p.name)[-1]
    ckpt = latest / 'checkpoints' / checkpoint_name
    if not ckpt.exists():
        raise FileNotFoundError(f'checkpoint missing: {ckpt}')
    return ckpt


def _require_checkpoint_path(value: object, field_name: str) -> Path:
    text = str(value).strip()
    if not text:
        raise ValueError(f'{field_name} 不能为空。')
    return Path(text)


def _resolve_model_checkpoint_from_json(ckpt_json_path: Path, payload: dict) -> Path | None:
    candidates: list[Path] = []
    raw = str(payload.get('model_checkpoint_path', '')).strip()
    if raw:
        candidates.append(Path(raw))
        candidates.append(ckpt_json_path.parent / Path(raw).name)
    candidates.append(ckpt_json_path.parent / 'sequence_policy.pt')
    candidates.append(ckpt_json_path.parent.parent / 'sequence_policy.pt')

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def _try_resume_sequence_training(
    *,
    checkpoint_json_path: Path,
    runtime_config_path: Path,
    run_root: Path,
) -> Path | None:
    try:
        from cchp_physical_env.core.config_loader import (
            build_env_config_from_overrides,
            build_training_options,
            load_env_overrides,
            load_training_overrides,
        )
        from cchp_physical_env.core.data import compute_training_statistics, load_exogenous_data
        from cchp_physical_env.policy.checkpoint import load_policy
        from cchp_physical_env.policy.trainer import (
            SequencePolicyTrainer,
            SequenceTrainerConfig,
        )

        training_options = build_training_options(load_training_overrides(runtime_config_path))
        policy_name = str(training_options.get('policy', 'rule')).strip().lower()
        adapter = str(training_options.get('sequence_adapter', 'rule')).strip().lower()
        if policy_name != 'sequence_rule' or adapter not in {'transformer', 'mamba'}:
            return None

        ckpt_payload = json.loads(checkpoint_json_path.read_text(encoding='utf-8'))
        model_ckpt = _resolve_model_checkpoint_from_json(checkpoint_json_path, ckpt_payload)
        if model_ckpt is None:
            print('resume skipped: sequence model checkpoint not found near', checkpoint_json_path)
            return None

        train_df = load_exogenous_data(Path('data/processed/cchp_main_15min_2024.csv'))
        train_stats = compute_training_statistics(train_df)
        env_config = build_env_config_from_overrides(load_env_overrides(runtime_config_path))

        trainer_cfg = SequenceTrainerConfig(
            policy_backbone=adapter,
            history_steps=int(training_options['history_steps']),
            episode_days=int(training_options['episode_days']),
            train_steps=int(training_options['train_steps']),
            batch_size=int(training_options['batch_size']),
            update_epochs=int(training_options['update_epochs']),
            lr=float(training_options['lr']),
            seed=int(training_options['seed']),
            device=str(training_options['device']),
        )

        trainer = SequencePolicyTrainer(
            train_df=train_df,
            train_statistics=train_stats,
            env_config=env_config,
            config=trainer_cfg,
            run_root=run_root,
        )

        payload = load_policy(checkpoint_path=model_ckpt, map_location=trainer.device)
        trainer.model.load_state_dict(payload['state_dict'], strict=True)
        summary = trainer.train()
        resumed_ckpt_json = Path(summary['checkpoint_json_path'])
        print('resume sequence training done. new checkpoint:', resumed_ckpt_json)
        return resumed_ckpt_json
    except Exception as error:
        print('resume sequence training failed, fallback to CLI train:', repr(error))
        return None


_run_cli(['summary', '--env-config', runtime_cfg_path.as_posix()])

from cchp_physical_env.core.config_loader import build_training_options, load_training_overrides
training_options = build_training_options(load_training_overrides(runtime_cfg_path))
policy_name = str(training_options.get('policy', 'rule')).strip().lower()
adapter_name = str(training_options.get('sequence_adapter', 'rule')).strip().lower()
sb3_enabled = bool(training_options.get('sb3_enabled', False))
dpar_enabled = bool(getattr(rc, 'dpar_enabled', False))
dpar_cfg = dict(getattr(rc, 'dpar_overrides', {}))

active_checkpoint_json: Path | None = None
run_root = Path(rc.run_root)
run_root.mkdir(parents=True, exist_ok=True)

if dpar_enabled:
    dqn_checkpoint = _require_checkpoint_path(dpar_cfg.get('dqn_checkpoint', ''), 'dpar_overrides.dqn_checkpoint')
    pafc_checkpoint = _require_checkpoint_path(dpar_cfg.get('pafc_checkpoint', ''), 'dpar_overrides.pafc_checkpoint')

    if rc.resume_enabled and rc.skip_train_when_resuming and resume_checkpoint_json is not None:
        print('resume DPAR checkpoint found, skip fresh hybrid assembly:', resume_checkpoint_json)
        active_checkpoint_json = resume_checkpoint_json
    else:
        train_args = [
            'hybrid-train',
            '--env-config', runtime_cfg_path.as_posix(),
            '--run-root', run_root.as_posix(),
            '--dqn-checkpoint', dqn_checkpoint.as_posix(),
            '--pafc-checkpoint', pafc_checkpoint.as_posix(),
            '--discrete-hold-steps', str(int(dpar_cfg.get('discrete_hold_steps', 1))),
            '--residual-scale', str(float(dpar_cfg.get('residual_scale', 1.0))),
            '--refine-action-keys', str(dpar_cfg.get('refine_action_keys', 'u_boiler')).strip() or 'u_boiler',
            '--model-source', str(dpar_cfg.get('model_source', 'best')).strip().lower() or 'best',
            '--device', str(rc.training_overrides['device']),
            '--seed', str(rc.training_overrides['seed']),
        ]
        refine_action_scales = str(dpar_cfg.get('refine_action_scales', '')).strip()
        if refine_action_scales:
            train_args.extend(['--refine-action-scales', refine_action_scales])
        if bool(dpar_cfg.get('stochastic_dqn', False)):
            train_args.append('--stochastic-dqn')
        if bool(dpar_cfg.get('use_safe_residual', False)):
            train_args.append('--use-safe-residual')
        _run_cli(train_args)
        active_checkpoint_json = _latest_checkpoint_from_run_root(run_root, checkpoint_name='hybrid_pafc_policy.json')
else:
    if (
        rc.resume_enabled
        and rc.resume_sequence_via_api
        and resume_checkpoint_json is not None
        and (not sb3_enabled)
        and policy_name == 'sequence_rule'
        and adapter_name in {'transformer', 'mamba'}
    ):
        active_checkpoint_json = _try_resume_sequence_training(
            checkpoint_json_path=resume_checkpoint_json,
            runtime_config_path=runtime_cfg_path,
            run_root=run_root,
        )

    if active_checkpoint_json is None:
        if rc.resume_enabled and rc.skip_train_when_resuming and resume_checkpoint_json is not None and (not sb3_enabled):
            print('resume checkpoint found, skip fresh train:', resume_checkpoint_json)
            active_checkpoint_json = resume_checkpoint_json
        else:
            _run_cli([
                'train',
                '--env-config', runtime_cfg_path.as_posix(),
                '--run-root', run_root.as_posix(),
            ])
            active_checkpoint_json = _latest_checkpoint_from_run_root(run_root, checkpoint_name='baseline_policy.json')

assert active_checkpoint_json is not None and active_checkpoint_json.exists(), active_checkpoint_json
print('checkpoint for eval:', active_checkpoint_json)

if dpar_enabled:
    eval_args = [
        'hybrid-eval',
        '--env-config', runtime_cfg_path.as_posix(),
        '--run-dir', active_checkpoint_json.parent.parent.as_posix(),
        '--checkpoint', active_checkpoint_json.as_posix(),
        '--seed', str(rc.training_overrides['seed']),
        '--device', str(rc.training_overrides['device']),
    ]
else:
    eval_args = [
        'eval',
        '--env-config', runtime_cfg_path.as_posix(),
        '--checkpoint', active_checkpoint_json.as_posix(),
        '--seed', str(rc.training_overrides['seed']),
    ]
    for key, value in rc.eval_cli_overrides.items():
        eval_args.extend([f'--{key.replace("_", "-")}', str(value)])
_run_cli(eval_args)


## 6) 输出检查

- `*_runtime_config.yaml` 是否生成
- `runs/.../checkpoints/baseline_policy.json` 是否存在
- `runs/.../eval/summary.json` 是否存在

In [ ]:
from pathlib import Path

runtime_cfg_candidates = sorted(Path('/kaggle/working').glob('*_runtime_config.yaml')) if Path('/kaggle/working').exists() else []
print('runtime configs:', [p.as_posix() for p in runtime_cfg_candidates])

run_root = Path(rc.run_root)
expected_checkpoint_name = 'hybrid_pafc_policy.json' if bool(getattr(rc, 'dpar_enabled', False)) else 'baseline_policy.json'
if run_root.exists():
    run_dirs = sorted([p for p in run_root.iterdir() if p.is_dir()], key=lambda p: p.name)
    print('run dirs:', [p.name for p in run_dirs[-5:]])
    if run_dirs:
        latest = run_dirs[-1]
        print('latest run:', latest)
        print('expected checkpoint:', latest / 'checkpoints' / expected_checkpoint_name)
        print('checkpoint exists:', (latest / 'checkpoints' / expected_checkpoint_name).exists())
        print('eval summary exists:', (latest / 'eval' / 'summary.json').exists())


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import time


def _safe_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


export_root = Path('kaggle_exports') / rc.run_prefix
export_root.mkdir(parents=True, exist_ok=True)

export_runtime_cfg = export_root / runtime_cfg_path.name
_safe_copy(runtime_cfg_path, export_runtime_cfg)

export_checkpoint_json = export_root / active_checkpoint_json.name
_safe_copy(active_checkpoint_json, export_checkpoint_json)

run_dir: Path | None = None
try:
    run_dir = active_checkpoint_json.parent.parent
except Exception:
    run_dir = None

export_eval_summary: Path | None = None
if run_dir is not None:
    eval_summary = run_dir / 'eval' / 'summary.json'
    if eval_summary.exists():
        export_eval_summary = export_root / 'eval_summary.json'
        _safe_copy(eval_summary, export_eval_summary)

export_checkpoints_dir = export_root / 'checkpoints'
if run_dir is not None:
    src_ckpt_dir = run_dir / 'checkpoints'
    if src_ckpt_dir.exists() and src_ckpt_dir.is_dir():
        if export_checkpoints_dir.exists():
            shutil.rmtree(export_checkpoints_dir)
        shutil.copytree(src_ckpt_dir, export_checkpoints_dir)

manifest = {
    'created_at_unix': int(time.time()),
    'run_prefix': rc.run_prefix,
    'dpar_enabled': bool(getattr(rc, 'dpar_enabled', False)),
    'runtime_config': export_runtime_cfg.as_posix(),
    'checkpoint_json': export_checkpoint_json.as_posix(),
    'eval_summary': export_eval_summary.as_posix() if export_eval_summary is not None else '',
    'run_dir': run_dir.as_posix() if run_dir is not None else '',
}
(export_root / 'export_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('export_root:', export_root.resolve())
print('export_runtime_cfg:', export_runtime_cfg)
print('export_checkpoint_json:', export_checkpoint_json)
print('export_eval_summary:', export_eval_summary)
print('export_checkpoints_dir:', export_checkpoints_dir if export_checkpoints_dir.exists() else None)
